In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.base import clone

import warnings
warnings.filterwarnings('ignore')

In [2]:
data_ace_24 = pd.read_csv("../data/Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("../data/Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("../data/Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("../data/Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [3]:
data_ace_24

,Dst,bx_gsm,by_gsm,bz_gsm,bt,proton_density,proton_speed,proton_temperature,doy_sin,doy_cos,...,proton_temperature_lag15,proton_temperature_lag16,proton_temperature_lag17,proton_temperature_lag18,proton_temperature_lag19,proton_temperature_lag20,proton_temperature_lag21,proton_temperature_lag22,proton_temperature_lag23,proton_temperature_lag24
datetime,,,,,,,,,,,,,,,,,,,,,
1997-10-22 00:00:00,2,-1.5369,-4.131,-3.5453,7.2194,32.4228,321.8729,44892.0,-0.9350748843515574,0.3544502230989869,...,27392.0,27161.0,29635.0,27439.0,25629.0,22869.0,17707.0,12445.0,20405.0,19301.0
1997-10-22 01:00:00,2,-3.5041,-6.6947,-3.1231,8.2642,35.3086,319.8263,37517.0,-0.9350748843515574,0.3544502230989869,...,26108.0,27392.0,27161.0,29635.0,27439.0,25629.0,22869.0,17707.0,12445.0,20405.0
1997-10-22 02:00:00,-4,-1.1892,-9.2528,0.7532,9.9372,29.9459,310.9227,41985.0,-0.9350748843515574,0.3544502230989869,...,26050.0,26108.0,27392.0,27161.0,29635.0,27439.0,25629.0,22869.0,17707.0,12445.0
1997-10-22 03:00:00,-7,-0.4474,-7.5079,6.4835,10.3952,25.9648,307.649,42621.0,-0.9350748843515574,0.3544502230989869,...,27376.0,26050.0,26108.0,27392.0,27161.0,29635.0,27439.0,25629.0,22869.0,17707.0
1997-10-22 04:00:00,-3,0.9936,-4.0181,7.0811,8.369,22.8004,299.1434,42575.0,-0.9350748843515574,0.3544502230989869,...,27391.0,27376.0,26050.0,26108.0,27392.0,27161.0,29635.0,27439.0,25629.0,22869.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-11 19:00:00,-45,5.4906,-5.0092,-5.4187,9.9549,2.2571,542.2711,26949.0,0.18809941761179758,0.9821499931752291,...,319120.0,297920.0,201370.0,350060.0,295810.0,230440.0,146490.0,164430.0,252890.0,115160.0
2026-01-11 20:00:00,-43,5.2535,-5.2342,-5.2958,9.7942,2.1076,536.2408,17706.0,0.18809941761179758,0.9821499931752291,...,299830.0,319120.0,297920.0,201370.0,350060.0,295810.0,230440.0,146490.0,164430.0,252890.0
2026-01-11 21:00:00,-48,3.6365,-7.5554,-3.4718,9.2759,1.3519,540.8746,25506.0,0.18809941761179758,0.9821499931752291,...,281180.0,299830.0,319120.0,297920.0,201370.0,350060.0,295810.0,230440.0,146490.0,164430.0


In [4]:
# a = data_ace_24.copy()
# d = data_discover_24.copy()

# split_date = "2022-01-01"

# # a_train = a.loc[:split_date]
# # d_train = d.loc[:split_date]

# # a_test = a.loc[split_date:]
# # d_test = d.loc[split_date:]

# overlap = a.index.intersection(d.index)

# ace_overlap = a.loc[overlap]

# a_train = ace_overlap.loc[:split_date]
# a_test = ace_overlap.loc[split_date:]

# len(a_train), len(a_test)
# соотношение 55% / 45%

In [4]:
def build_models_adaptation():
    models = {
        "linreg": LinearRegression(),
        "boost": LGBMRegressor(
            n_estimators=1000,
            reg_alpha=0.7,
            reg_lambda=0.7,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose=-1),
        "pls": PLSRegression(n_components=100,
            scale=True,
            max_iter=2000)}
    return models

def adapt_discover_to_ace(ace_df, disc_df, future_lags, split_date="2021-01-01", adapt_model_name="linreg"):
    """
    Адаптация discover -> ace
    1. L обучается только на пересечении тренировочных данных ace и discover
    2. Тест discover адаптируется в домен данных ace
    3. Из всех данных возвращается только discover_test_adapted для дальнейшего предсказания
    """
    ace = ace_df.sort_index()
    disc = disc_df.sort_index()
    split_date = pd.Timestamp(split_date)

    ace_train = ace.loc[:split_date]
    disc_train = disc.loc[:split_date]
    disc_test = disc.loc[split_date:]   # адаптируем только тест discover

    # Пересечение индексов внутри train
    overlap_idx = ace_train.index.intersection(disc_train.index)

    ace_overlap = ace_train.loc[overlap_idx]
    disc_overlap = disc_train.loc[overlap_idx]

    feature_cols = [c for c in ace.columns if c not in future_lags]
    
    # Масштабирование
    sc_ace = StandardScaler().fit(ace_overlap[feature_cols])
    sc_disc = StandardScaler().fit(disc_overlap[feature_cols])

    X_ace = sc_ace.transform(ace_overlap[feature_cols])
    X_disc = sc_disc.transform(disc_overlap[feature_cols])

   # Выбор модели адаптации L: discover -> ace
    models = build_models_adaptation()
    
    if adapt_model_name not in models:
        raise ValueError(f"Неизвестная модель: {adapt_model_name}. Доступны: {list(models.keys())}")
    
    L = models[adapt_model_name]
    
    # Используем, так как таргет многомерный
    if adapt_model_name == "boost":
        from sklearn.multioutput import MultiOutputRegressor
        L = MultiOutputRegressor(L)

    L.fit(X_disc, X_ace)

    X_disc_test = sc_disc.transform(disc_test[feature_cols])            # Нормировка тестового набора данных discover
    X_disc_test_adapted = L.predict(X_disc_test)                        # Применение обученной модели адаптации
    X_disc_test_adapted = sc_ace.inverse_transform(X_disc_test_adapted) # Приводим новые адаптированные данные к ненормированному виду

    disc_test_adapted = pd.DataFrame(X_disc_test_adapted, index=disc_test.index, columns=feature_cols)

    # Целевые переменные возвращаются обратно неизменёнными
    for col in future_lags:
        disc_test_adapted[col] = disc_test[col]

    return disc_test_adapted, L, sc_disc, sc_ace

In [5]:
def cross_adapt_domain(ace_df, disc_df, random_state=42,
                       wind_model_type='LGBM'):
    """
    Обучение доменной адаптации:
    - для магнитного поля (field_features) — линейная регрессия
    - для параметров солнечного ветра (wind_features) — LGBM или MLP
    """
    field_features = ace_df.filter(regex=r'^(Dst|bx_gsm|by_gsm|bz_gsm|bt)(_lag\d+)?$').columns.tolist()
    wind_features = ace_df.filter(regex=r'^(proton_density|proton_speed|proton_temperature)(_lag\d+)?$').columns.tolist()

    # модель для полей
    lin = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])

    # модель для ветра
    if wind_model_type == 'LGBM':
        wind_model = Pipeline([
            ("scaler", StandardScaler()),
            ("boost", LGBMRegressor(
                n_estimators=1000,
                reg_alpha=0.7,
                reg_lambda=0.7,
                max_depth=3,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbose=-1))
        ])
    else:
        wind_model = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPRegressor(
                hidden_layer_sizes=(8, 4),
                solver='adam',
                alpha=1e-3,
                batch_size=64,
                learning_rate_init=5e-4,
                early_stopping=True,
                validation_fraction=0.15,
                max_iter=1500,
                random_state=random_state,
                verbose=False))
        ])

    disc_adapted = disc_df.copy()

    def make_wind_model():
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", clone(wind_model))
        ])

    disc_adapted = disc_df.copy()

    for col in field_features:
        if col not in ace_df.columns or col not in disc_df.columns:
            continue
        lin.fit(ace_df[[col]], ace_df[col])
        disc_adapted[col] = lin.predict(disc_df[[col]])

    wind_cols = [c for c in wind_features if c in ace_df.columns and c in disc_df.columns]

    if wind_cols:
        for target_col in wind_cols:
            wind_model = make_wind_model()
            X_ace = ace_df[wind_cols]
            y_ace = ace_df[target_col]

            wind_model.fit(X_ace, y_ace)

            X_disc = disc_df[wind_cols]
            disc_adapted[target_col] = wind_model.predict(X_disc)

    return disc_adapted

In [6]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("boost", LGBMRegressor(
            boosting_type='gbdt',
            num_leaves=93,
            max_depth=5,
            learning_rate=0.014357868416776678,
            n_estimators=1160,
            # subsample_for_bin=200000,
            objective=None,
            class_weight=None,
            min_split_gain=0.0,
            min_child_weight=0.001,
            min_child_samples=25,
            subsample=0.5606239658637814,
            subsample_freq=0,
            colsample_bytree=0.9879825765791656,
            reg_alpha=0.022687699993507067,
            reg_lambda=0.005915305250069859,
            random_state=random_state,
            n_jobs=None,
            importance_type='split',
            metric='rmse',
            # early_stopping_rounds=50,
            verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(12,),
            solver='adam',
            alpha=1e-4,
            batch_size=32,
            learning_rate_init=1e-3,
            early_stopping=True,
            validation_fraction=0.1,
            max_iter=1500,
            random_state=random_state))
    ])
    return models
    
# models = build_models()

def evaluate_M_A(ace_df, disc_test_adapted_df, split_date, target, future_lags,
                      adaptation_method='linreg', delays='24h',
                      random_seed=42, verbose=True):
    """
    Модель M_A обучается на ace_train и тестируется на адаптированном discover_test
    """
    models = build_models(random_state=random_seed)
    split_date = pd.Timestamp(split_date)

    ace_train = ace_df.loc[:split_date]

    feature_cols = [c for c in ace_train.columns if c not in future_lags]

    X_train = ace_train[feature_cols].values
    y_train = ace_train[target].values

    X_test = disc_test_adapted_df[feature_cols].values
    y_test = disc_test_adapted_df[target].values

    run_results = []

    # if verbose:
    #     print(f"\n=== M_A | target={target} | adapt={adaptation_method} | seed={random_seed} ===")

    for name, model in models.items():
        try:
            if name == 'LGBM':
                X_tr, X_val, y_tr, y_val = train_test_split(
                    X_train, y_train, test_size=0.1, random_state=random_seed)

                model.fit(
                    X_tr, y_tr,
                    boost__eval_set=[(X_val, y_val)],
                    boost__eval_metric='l2'
                )
            else:
                model.fit(X_train, y_train)
        
            y_pred = model.predict(X_test)
            
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            mae  = mean_absolute_error(y_test, y_pred)
            r2   = r2_score(y_test, y_pred)

            # if verbose:
            #     print(f"{name:8s} seed={random_seed}: rmse={rmse:.4f}, mae={mae:.4f}, r2={r2:.4f}")

            run_results.append({
                'target': target,
                'delays': delays,
                'adaptation_method': adaptation_method,
                'forecast_model': name,
                'seed': random_seed,
                'RMSE': rmse,
                'MAE': mae,
                'R2': r2
            })
        except Exception as e:
            print(f"{name:8s} seed={random_seed} -> Ошибка обучения: {e}")

    return run_results


In [7]:
def evaluate_M_A_with_error(ace_df, disc_test_adapted_df, split_date, target, future_lags,
                            adaptation_method='linreg', delays='24h',
                            n_seeds=5, base_seed=42,
                            results_list=None, verbose=True):
    if results_list is None:
        results_list = []

    base_seed_single = base_seed
    single_run = evaluate_M_A(
        ace_df, disc_test_adapted_df, split_date, target, future_lags,
        adaptation_method=adaptation_method,
        delays=delays,
        random_seed=base_seed_single,
        verbose=verbose
    )
    df_single = pd.DataFrame(single_run)

    seeds = [base_seed + i for i in range(n_seeds)]
    stochastic_models = ['LGBM', 'MLP']

    all_runs_stochastic = []
    for seed in seeds:
        run_results = evaluate_M_A(
            ace_df, disc_test_adapted_df, split_date, target, future_lags,
            adaptation_method=adaptation_method,
            delays=delays,
            random_seed=seed,
            verbose=False)
        
        run_results = [r for r in run_results if r['forecast_model'] in stochastic_models]
        all_runs_stochastic.extend(run_results)

    df_stoch = pd.DataFrame(all_runs_stochastic)

    records = []

    linear_models = ['Linear', 'Ridge', 'Lasso']
    df_linear = df_single[df_single['forecast_model'].isin(linear_models)]

    for _, row in df_linear.iterrows():
        records.append({
            'target': row['target'],
            'delays': delays,
            'adaptation_method': adaptation_method,
            'forecast_model': row['forecast_model'],
            'RMSE': row['RMSE'],
            'RMSE_std': 0.0,
            'MAE': row['MAE'],
            'MAE_std': 0.0,
            'R2': row['R2'],
            'R2_std': 0.0,
            'n_seeds': 1,
            'base_seed': base_seed_single
        })

    if not df_stoch.empty:
        group_cols = ['target', 'adaptation_method', 'forecast_model', 'delays']
        agg = df_stoch.groupby(group_cols).agg({
            'RMSE': ['mean', 'std'],
            'MAE':  ['mean', 'std'],
            'R2':   ['mean', 'std']
        }).reset_index()

        agg.columns = ['target', 'adaptation_method', 'forecast_model', 'delays',
                       'RMSE_mean', 'RMSE_std',
                       'MAE_mean',  'MAE_std',
                       'R2_mean',   'R2_std']

        for _, row in agg.iterrows():
            records.append({
                'target': row['target'],
                'delays': row['delays'],
                'adaptation_method': row['adaptation_method'],
                'forecast_model': row['forecast_model'],
                'RMSE': row['RMSE_mean'],
                'RMSE_std':  row['RMSE_std'],
                'MAE':  row['MAE_mean'],
                'MAE_std':   row['MAE_std'],
                'R2':   row['R2_mean'],
                'R2_std':    row['R2_std'],
                'n_seeds':   n_seeds,
                'base_seed': base_seed
            })

    for rec in records:
        results_list.append(rec)

    df_all = pd.concat([df_single, df_stoch], ignore_index=True)

    agg_df = pd.DataFrame(records)

    if verbose:
        for _, row in agg_df.iterrows():
            print(f"{row['forecast_model']:8s}: "
                  f"RMSE = {row['RMSE']:.4f} ± {row['RMSE_std']:.4f}, "
                  f"MAE = {row['MAE']:.4f} ± {row['MAE_std']:.4f}, "
                  f"R2 = {row['R2']:.4f} ± {row['R2_std']:.4f}")

    return df_all, agg_df, results_list

In [8]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

cutoff_date = pd.Timestamp('2023-12-31 23:59:59')

data_ace_24_copy = data_ace_24_copy[data_ace_24_copy.index <= cutoff_date]
data_discover_24_copy = data_discover_24_copy[data_discover_24_copy.index <= cutoff_date]

data_ace_af_copy = data_ace_af_copy[data_ace_af_copy.index <= cutoff_date]
data_discover_af_copy = data_discover_af_copy[data_discover_af_copy.index <= cutoff_date]

# 1. Задание переменных для адаптации
split_date = "2022-01-01"
future_lags = [f'Dst_plus{i}' for i in range(1, 25)] # все сдвиги во времени вперед (задаются при обработке данных)
# targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3', 'Dst_plus6', 'Dst_plus12', 'Dst_plus24'] # таргеты, на которые делаем прогнозы
targets = ['Dst_plus1', 'Dst_plus3', 'Dst_plus6']

In [9]:
# 2.1. Доменная адаптация с помощью моделей линейной регрессии (linreg), градиентного бустинга (boost) и метода проекций на латентные структуры (pls)
disc_adapted_lin_24, L_lin_24, sd_lin_24, sa_lin_24 = adapt_discover_to_ace(
    data_ace_24_copy,
    data_discover_24_copy,
    future_lags,
    split_date,
    adapt_model_name="linreg"
)

disc_adapted_gbr_24, L_gbr_24, sd_gbr_24, sa_gbr_24 = adapt_discover_to_ace(
    data_ace_24_copy,
    data_discover_24_copy,
    future_lags,
    split_date,
    adapt_model_name="boost"
)

disc_adapted_pls_24, L_pls_24, sd_pls_24, sa_pls_24 = adapt_discover_to_ace(
    data_ace_24_copy,
    data_discover_24_copy,
    future_lags,
    split_date,
    adapt_model_name="pls"
)

In [10]:
disc_adapted_lin_af, L_lin_af, sd_lin_af, sa_lin_af = adapt_discover_to_ace(
    data_ace_af_copy,
    data_discover_af_copy,
    future_lags,
    split_date,
    adapt_model_name="linreg"
)

disc_adapted_gbr_af, L_gbr_af, sd_gbr_af, sa_gbr_af = adapt_discover_to_ace(
    data_ace_af_copy,
    data_discover_af_copy,
    future_lags,
    split_date,
    adapt_model_name="boost"
)

disc_adapted_pls_af, L_pls_af, sd_pls_af, sa_pls_af = adapt_discover_to_ace(
    data_ace_af_copy,
    data_discover_af_copy,
    future_lags,
    split_date,
    adapt_model_name="pls"
)

In [11]:
results_list = []

In [12]:
print("\n==== M_A: Depth - 24h ====")

print("\n==== Adaptation - linreg ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_24_copy,
        disc_test_adapted_df=disc_adapted_lin_24,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-linreg',
        delays='24h',
        n_seeds=3,
        base_seed=42,
        results_list=results_list)

print("\n==== Adaptation - lgbm ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_24_copy,
        disc_test_adapted_df=disc_adapted_gbr_24,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-lgbm',
        delays='24h',
        n_seeds=3,
        base_seed=42,
        results_list=results_list)

print("\n==== Adaptation - pls ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_24_copy,
        disc_test_adapted_df=disc_adapted_pls_24,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-pls',
        delays='24h',
        n_seeds=3,
        base_seed=42,
        results_list=results_list)


==== M_A: Depth - 24h ====

==== Adaptation - linreg ====

==== Forecast of DST_PLUS1 ====
Linear  : RMSE = 3.3349 ± 0.0000, MAE = 2.3535 ± 0.0000, R2 = 0.9675 ± 0.0000
Ridge   : RMSE = 3.3349 ± 0.0000, MAE = 2.3535 ± 0.0000, R2 = 0.9675 ± 0.0000
Lasso   : RMSE = 3.3354 ± 0.0000, MAE = 2.3543 ± 0.0000, R2 = 0.9674 ± 0.0000
LGBM    : RMSE = 3.3370 ± 0.0356, MAE = 2.3184 ± 0.0059, R2 = 0.9674 ± 0.0007
MLP     : RMSE = 3.2247 ± 0.0281, MAE = 2.3085 ± 0.0237, R2 = 0.9696 ± 0.0005

==== Forecast of DST_PLUS3 ====
Linear  : RMSE = 6.6913 ± 0.0000, MAE = 4.7789 ± 0.0000, R2 = 0.8689 ± 0.0000
Ridge   : RMSE = 6.6913 ± 0.0000, MAE = 4.7790 ± 0.0000, R2 = 0.8689 ± 0.0000
Lasso   : RMSE = 6.6914 ± 0.0000, MAE = 4.7788 ± 0.0000, R2 = 0.8689 ± 0.0000
LGBM    : RMSE = 6.3917 ± 0.0122, MAE = 4.5284 ± 0.0038, R2 = 0.8803 ± 0.0005
MLP     : RMSE = 6.4300 ± 0.0368, MAE = 4.5898 ± 0.0354, R2 = 0.8789 ± 0.0014

==== Forecast of DST_PLUS6 ====
Linear  : RMSE = 9.9278 ± 0.0000, MAE = 6.9026 ± 0.0000, R2 = 

In [13]:
print(f"\n==== Depth - autocorrelation function ====")

print("\n==== Adaptation - linreg ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_af_copy,
        disc_test_adapted_df=disc_adapted_lin_af,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-linreg',
        delays='auto_func',
        n_seeds=5,
        base_seed=42,
        results_list=results_list)

print("\n==== Adaptation - lgbm ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_af_copy,
        disc_test_adapted_df=disc_adapted_gbr_af,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-lgbm',
        delays='auto_func',
        n_seeds=5,
        base_seed=42,
        results_list=results_list)

print("\n==== Adaptation - pls ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_af_copy,
        disc_test_adapted_df=disc_adapted_pls_af,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-pls',
        delays='auto_func',
        n_seeds=5,
        base_seed=42,
        results_list=results_list)


==== Depth - autocorrelation function ====

==== Adaptation - linreg ====

==== Forecast of DST_PLUS1 ====
Linear  : RMSE = 3.3563 ± 0.0000, MAE = 2.3723 ± 0.0000, R2 = 0.9669 ± 0.0000
Ridge   : RMSE = 3.3563 ± 0.0000, MAE = 2.3723 ± 0.0000, R2 = 0.9669 ± 0.0000
Lasso   : RMSE = 3.3565 ± 0.0000, MAE = 2.3731 ± 0.0000, R2 = 0.9669 ± 0.0000
LGBM    : RMSE = 3.3112 ± 0.0153, MAE = 2.3201 ± 0.0031, R2 = 0.9677 ± 0.0003
MLP     : RMSE = 3.2430 ± 0.0310, MAE = 2.3302 ± 0.0284, R2 = 0.9691 ± 0.0006

==== Forecast of DST_PLUS3 ====
Linear  : RMSE = 6.7725 ± 0.0000, MAE = 4.8617 ± 0.0000, R2 = 0.8650 ± 0.0000
Ridge   : RMSE = 6.7725 ± 0.0000, MAE = 4.8617 ± 0.0000, R2 = 0.8650 ± 0.0000
Lasso   : RMSE = 6.7723 ± 0.0000, MAE = 4.8614 ± 0.0000, R2 = 0.8651 ± 0.0000
LGBM    : RMSE = 6.3423 ± 0.0110, MAE = 4.5615 ± 0.0032, R2 = 0.8816 ± 0.0004
MLP     : RMSE = 6.4164 ± 0.0275, MAE = 4.6232 ± 0.0340, R2 = 0.8789 ± 0.0010

==== Forecast of DST_PLUS6 ====
Linear  : RMSE = 10.0136 ± 0.0000, MAE = 7.044

In [14]:
disc_adapted_cross_lgbm_24 = cross_adapt_domain(data_ace_24_copy, data_discover_24_copy, 
                                               wind_model_type='LGBM')
disc_adapted_cross_mlp_24 = cross_adapt_domain(data_ace_24_copy, data_discover_24_copy, 
                                              wind_model_type='MLP')

In [15]:
disc_adapted_cross_lgbm_af = cross_adapt_domain(data_ace_af_copy, data_discover_af_copy, 
                                               wind_model_type='LGBM')
disc_adapted_cross_mlp_af = cross_adapt_domain(data_ace_af_copy, data_discover_af_copy, 
                                              wind_model_type='MLP')

In [16]:
print("\n==== M_A: Depth - 24h ====")

print("\n==== Adaptation - cross-linreg+lgbm ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_24_copy,
        disc_test_adapted_df=disc_adapted_cross_lgbm_24,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-cross-linreg+lgbm',
        delays='24h',
        n_seeds=3,
        base_seed=42,
        results_list=results_list)

print("\n==== Adaptation - cross-linreg+mlp ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_24_copy,
        disc_test_adapted_df=disc_adapted_cross_mlp_24,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-cross-linreg+mlp',
        delays='24h',
        n_seeds=3,
        base_seed=42,
        results_list=results_list)


==== M_A: Depth - 24h ====

==== Adaptation - cross-linreg+lgbm ====

==== Forecast of DST_PLUS1 ====
Linear  : RMSE = 2.9159 ± 0.0000, MAE = 2.0626 ± 0.0000, R2 = 0.9636 ± 0.0000
Ridge   : RMSE = 2.9158 ± 0.0000, MAE = 2.0626 ± 0.0000, R2 = 0.9636 ± 0.0000
Lasso   : RMSE = 2.9037 ± 0.0000, MAE = 2.0556 ± 0.0000, R2 = 0.9639 ± 0.0000
LGBM    : RMSE = 2.9076 ± 0.0076, MAE = 2.0321 ± 0.0032, R2 = 0.9638 ± 0.0002
MLP     : RMSE = 2.9905 ± 0.0678, MAE = 2.1258 ± 0.0526, R2 = 0.9617 ± 0.0018

==== Forecast of DST_PLUS3 ====
Linear  : RMSE = 5.8890 ± 0.0000, MAE = 4.2367 ± 0.0000, R2 = 0.8513 ± 0.0000
Ridge   : RMSE = 5.8890 ± 0.0000, MAE = 4.2367 ± 0.0000, R2 = 0.8513 ± 0.0000
Lasso   : RMSE = 5.8867 ± 0.0000, MAE = 4.2345 ± 0.0000, R2 = 0.8514 ± 0.0000
LGBM    : RMSE = 5.9110 ± 0.0113, MAE = 4.2606 ± 0.0102, R2 = 0.8502 ± 0.0006
MLP     : RMSE = 5.8774 ± 0.1431, MAE = 4.2410 ± 0.1217, R2 = 0.8519 ± 0.0073

==== Forecast of DST_PLUS6 ====
Linear  : RMSE = 8.4423 ± 0.0000, MAE = 5.9266 ± 0.

In [17]:
print("\n==== M_A: Depth - autocorrelation function ====")

print("\n==== Adaptation - cross-linreg+lgbm ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_af_copy,
        disc_test_adapted_df=disc_adapted_cross_lgbm_af,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-cross-linreg+lgbm',
        delays='auto_func',
        n_seeds=3,
        base_seed=42,
        results_list=results_list)

print("\n==== Adaptation - cross-linreg+mlp ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    df_all, df_sum, results_list = evaluate_M_A_with_error(
        ace_df=data_ace_af_copy,
        disc_test_adapted_df=disc_adapted_cross_mlp_af,
        split_date=split_date,
        target=target_col,
        future_lags=future_lags,
        adaptation_method='disc-ace-cross-linreg+mlp',
        delays='auto_func',
        n_seeds=3,
        base_seed=42,
        results_list=results_list)


==== M_A: Depth - autocorrelation function ====

==== Adaptation - cross-linreg+lgbm ====

==== Forecast of DST_PLUS1 ====
Linear  : RMSE = 2.9270 ± 0.0000, MAE = 2.0732 ± 0.0000, R2 = 0.9633 ± 0.0000
Ridge   : RMSE = 2.9269 ± 0.0000, MAE = 2.0731 ± 0.0000, R2 = 0.9633 ± 0.0000
Lasso   : RMSE = 2.9150 ± 0.0000, MAE = 2.0665 ± 0.0000, R2 = 0.9636 ± 0.0000
LGBM    : RMSE = 2.8913 ± 0.0043, MAE = 2.0324 ± 0.0026, R2 = 0.9642 ± 0.0001
MLP     : RMSE = 3.0002 ± 0.0778, MAE = 2.1534 ± 0.0561, R2 = 0.9614 ± 0.0020

==== Forecast of DST_PLUS3 ====
Linear  : RMSE = 5.9310 ± 0.0000, MAE = 4.2725 ± 0.0000, R2 = 0.8492 ± 0.0000
Ridge   : RMSE = 5.9309 ± 0.0000, MAE = 4.2725 ± 0.0000, R2 = 0.8492 ± 0.0000
Lasso   : RMSE = 5.9286 ± 0.0000, MAE = 4.2703 ± 0.0000, R2 = 0.8493 ± 0.0000
LGBM    : RMSE = 6.0037 ± 0.0717, MAE = 4.3627 ± 0.0486, R2 = 0.8454 ± 0.0037
MLP     : RMSE = 5.8180 ± 0.0823, MAE = 4.2045 ± 0.0813, R2 = 0.8549 ± 0.0041

==== Forecast of DST_PLUS6 ====
Linear  : RMSE = 8.4735 ± 0.00

In [18]:
results_df = pd.DataFrame(results_list)

In [19]:
results_df

,target,delays,adaptation_method,forecast_model,RMSE,RMSE_std,MAE,MAE_std,R2,R2_std,n_seeds,base_seed
0,Dst_plus1,24h,disc-ace-linreg,Linear,3.334871,0.000000,2.353491,0.000000,0.967456,0.000000,1,42
1,Dst_plus1,24h,disc-ace-linreg,Ridge,3.334874,0.000000,2.353517,0.000000,0.967456,0.000000,1,42
2,Dst_plus1,24h,disc-ace-linreg,Lasso,3.335384,0.000000,2.354300,0.000000,0.967446,0.000000,1,42
3,Dst_plus1,24h,disc-ace-linreg,LGBM,3.336954,0.035612,2.318413,0.005863,0.967413,0.000693,3,42
4,Dst_plus1,24h,disc-ace-linreg,MLP,3.224679,0.028104,2.308469,0.023681,0.969570,0.000531,3,42
...,...,...,...,...,...,...,...,...,...,...,...,...
145,Dst_plus6,auto_func,disc-ace-cross-linreg+mlp,Linear,8.538802,0.000000,5.995968,0.000000,0.687398,0.000000,1,42
146,Dst_plus6,auto_func,disc-ace-cross-linreg+mlp,Ridge,8.538800,0.000000,5.995961,0.000000,0.687398,0.000000,1,42
147,Dst_plus6,auto_func,disc-ace-cross-linreg+mlp,Lasso,8.536663,0.000000,5.994326,0.000000,0.687555,0.000000,1,42
148,Dst_plus6,auto_func,disc-ace-cross-linreg+mlp,LGBM,8.670519,0.041877,6.172136,0.019091,0.677675,0.003114,3,42


In [21]:
results_df.to_excel("../results/models-adaptation-discover-to-ace_2022-01-01 (22-24).xlsx", index=False)